# 01 — Select and Load Sample

Lock one real DB sample for the experiment and inspect why it is suitable.


In [ ]:
from pathlib import Path
import hashlib
import json
import logging
import os
import sys
from typing import Any

import yaml
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "code_base").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
MODULE_ROOT = PROJECT_ROOT / "code_base" / "fea_cad_one_sample"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

API_ENV_PATH = MODULE_ROOT / "src" / "api.env"


def _load_api_env(path: Path) -> dict[str, str]:
    env_values: dict[str, str] = {}
    if not path.exists():
        return env_values
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip("'").strip('"')
        if key and value:
            env_values[key] = value
            os.environ.setdefault(key, value)
    return env_values


api_env_values = _load_api_env(API_ENV_PATH)

from src import interfaces as api

logging.basicConfig(level=logging.INFO, format="%(name)s | %(levelname)s | %(message)s")

sample_id = "00689964"
selection_source = "dataset"  # Change to "db" to query the live database sample loader instead.
connection_string = os.environ.get("CAD_DB_CONNECTION_STRING")
dataset_sample_root = MODULE_ROOT / "outputs" / f"sample_{sample_id}"
dataset_original_dir = dataset_sample_root / "01_dataset_original"
experiment_dir = MODULE_ROOT / "experiment_config"
experiment_dir.mkdir(parents=True, exist_ok=True)
fixed_sample_path = experiment_dir / "fixed_sample.yaml"
selection_json_path = MODULE_ROOT / "outputs" / f"sample_{sample_id}" / "sample_selection.json"

print("[STAGE] Setup")
print(f"  → module root   : {MODULE_ROOT}")
print(f"  → fixed sample  : {sample_id}")
print(f"  → selection     : {selection_json_path}")
print(f"  → selection mode: {selection_source}")
print(f"  → dataset root  : {dataset_sample_root}")
print(f"  → dataset dir   : {dataset_original_dir}")
print(f"  → api.env       : {API_ENV_PATH}")
print(f"  → api.env keys  : {sorted(api_env_values)}")

In [ ]:
print("[STAGE] Load sample")
sample = api.load_selected_sample(
    module_root=MODULE_ROOT,
    sample_id=sample_id,
    selection_source=selection_source,
    connection_string=connection_string,
)
prompt_preview = "\n".join(sample.prompt.splitlines()[:8])
code_text = sample.ground_truth_code
code_hash = hashlib.sha256(code_text.encode("utf-8")).hexdigest()
fea_terms = ["bracket", "support", "clamp", "mount", "mounting plate", "connector", "cantilever"]
matched_terms = [term for term in fea_terms if term in (sample.prompt + "\n" + code_text).lower()]

print(f"  → sample id : {sample.sample_id}")
print(f"  → prompt    : {sample.prompt_variant}")
print(f"  → source    : {sample.source}")
print(f"  → code len  : {len(code_text)}")
print(f"  → code hash : {code_hash}")
print(f"  → matched   : {matched_terms if matched_terms else 'none'}")
print("")
display(Markdown("## Prompt preview"))
display(Markdown(f"```text\n{prompt_preview}\n```"))
assert sample.sample_id == sample_id
assert bool(sample.prompt.strip())
assert bool(code_text.strip())
compile(code_text, "<selected_sample_code>", "exec")
print("  ✓ sample data exists and is executable")

In [ ]:
print("[STAGE] Save fixed sample config")
fixed_sample_payload = {
    "sample_id": sample.sample_id,
    "prompt_variant": sample.prompt_variant,
    "source": sample.source,
    "code_hash_sha256": code_hash,
    "selection_mode": sample.metadata.get("selection_mode", selection_source),
    "load_source": sample.metadata.get("load_source", selection_source),
    "dataset_root": sample.metadata.get("dataset_root"),
    "dataset_original_dir": sample.metadata.get("dataset_original_dir"),
    "reason": (
        "Loaded from the local dataset artifacts under outputs/sample_00689964/01_dataset_original/ "
        "instead of querying the live database."
    ),
}
if selection_source == "db":
    fixed_sample_payload["reason"] = "Loaded from the live database using the shared notebook loader helper."
fixed_sample_path.write_text(yaml.safe_dump(fixed_sample_payload, sort_keys=True), encoding="utf-8")
selection_json_path.write_text(json.dumps(fixed_sample_payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(f"  → fixed config : {fixed_sample_path}")
print(f"  → selection js : {selection_json_path}")
assert fixed_sample_path.exists()
assert selection_json_path.exists()
assert fixed_sample_payload["sample_id"]
assert fixed_sample_payload["code_hash_sha256"]
print("  ✓ sample selection locked for later notebooks")

## What this notebook proves

- A real DB sample was selected and locked.
- The prompt is readable.
- The original CAD code exists and is executable.
